# GCS to Artifact Import Verification

This notebook verifies the functionality of the `ImportGcsToArtifactTool`, which allows the agent to register GCS objects as session artifacts for multi-modal analysis.

In [ ]:
import sys
import asyncio
from unittest.mock import MagicMock, AsyncMock

# Ensure project root is in path
sys.path.append("../..")

from agent.core_agent.internal_tools.artifact_tools import ImportGcsToArtifactTool
from google.adk.tools import ToolContext
from google.genai import types

print("Setup complete.")

## 1. Tool Logic Verification

We will mock the `ToolContext.save_artifact` method and verify that the tool correctly prepares the `FileData` part and handles the import flow.

In [ ]:
async def test_tool_import():
    tool = ImportGcsToArtifactTool()
    
    # Mock ToolContext
    mock_context = MagicMock(spec=ToolContext)
    mock_context.save_artifact = AsyncMock(return_value=1) # Returns version 1
    
    # Test Case 1: Import a PDF
    args = {
        "gcs_uri": "gs://my-bucket/documents/report.pdf",
        "artifact_name": "quarterly_report"
    }
    
    print(f"Testing import of: {args['gcs_uri']}")
    result = await tool.run_async(args=args, tool_context=mock_context)
    
    print(f"Status: {result['execution_status']}")
    print(f"Artifact ID: {result['artifact_id']}")
    print(f"Message: {result['execution_message']}")
    
    # Verify save_artifact call
    mock_context.save_artifact.assert_called_once()
    call_kwargs = mock_context.save_artifact.call_args.kwargs
    
    assert call_kwargs['filename'] == "quarterly_report"
    artifact_part = call_kwargs['artifact']
    assert artifact_part.file_data.file_uri == args['gcs_uri']
    assert artifact_part.file_data.mime_type == "application/pdf"
    
    print("\nSuccess: Tool correctly prepared the artifact registration!")

await test_tool_import()

## 2. Integration with GCS Metadata

In a real scenario, the agent first calls `read_object` to get the metadata. Here we simulate that flow.

In [ ]:
async def test_integration_flow():
    # Simulated read_object response
    read_response = {
        "gcs_uri": "gs://ai_agent_landing_zone/user_upload_123.jpg",
        "content_type": "image/jpeg",
        "metadata": {"uploader": "user@example.com"}
    }
    
    tool = ImportGcsToArtifactTool()
    mock_context = MagicMock(spec=ToolContext)
    mock_context.save_artifact = AsyncMock(return_value=1)
    
    # Agent uses the gcs_uri from read_object
    result = await tool.run_async(
        args={"gcs_uri": read_response["gcs_uri"]}, 
        tool_context=mock_context
    )
    
    print(f"Imported image: {result['artifact_id']}")
    print(f"Status: {result['execution_status']}")

await test_integration_flow()